<a href="https://colab.research.google.com/github/AyeshaRashid345/Bano_Qabil_AI_Assistant./blob/main/Bano_Qabil_AI_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Bano Qabil AI Assistant")
print("Assignment 02")
print("Starting RAG Project...")

Bano Qabil AI Assistant
Assignment 02
Starting RAG Project...


In [ ]:
!pip install -q google-genai==2.12.1 google-auth==2.49.0 requests==2.32.4 pinecone beautifulsoup4

In [ ]:
from google.colab import userdata
from google import genai
from pinecone import Pinecone

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)

print("Gemini connected!")
print("Pinecone connected!")

Gemini connected!
Pinecone connected!


In [ ]:
import requests
from bs4 import BeautifulSoup

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
BASE_URL = "https://banoqabil.pk"

response = requests.get(
    BASE_URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("Website connected successfully!")
print("Page title:", soup.title.get_text(strip=True) if soup.title else "No title")

Website connected successfully!
Page title: Bano Qabil - Free IT Training Program


In [ ]:
from urllib.parse import urljoin, urlparse, urldefrag
import time

visited_urls = set()
pages_data = []

def scrape_page(url):
    try:
        r = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=20
        )
        r.raise_for_status()

        page = BeautifulSoup(r.text, "html.parser")

        for tag in page(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        text = page.get_text(separator=" ", strip=True)

        if len(text) > 100:
            pages_data.append({
                "url": url,
                "text": text
            })

        return page

    except Exception as e:
        print("Page skipped:", url, str(e))
        return None


homepage = scrape_page(BASE_URL)

if homepage:
    for link in homepage.find_all("a", href=True):
        full_url = urldefrag(urljoin(BASE_URL, link["href"]))[0]
        parsed = urlparse(full_url)

        if (
            parsed.netloc == urlparse(BASE_URL).netloc
            and parsed.scheme in ["http", "https"]
            and full_url not in visited_urls
        ):
            visited_urls.add(full_url)

        if len(visited_urls) >= 20:
            break

for url in list(visited_urls):
    scrape_page(url)
    time.sleep(0.5)

print("Total pages collected:", len(pages_data))

Total pages collected: 3


In [ ]:
for i, page in enumerate(pages_data, 1):
    print(f"\nPage {i}:")
    print("URL:", page["url"])
    print("Text length:", len(page["text"]))
    print("Preview:", page["text"][:200])
    print("-" * 50)


Page 1:
URL: https://banoqabil.pk
Text length: 5259
Preview: Bano Qabil - Free IT Training Program Start Your Journey Today Join thousands learning tech skills for free Register Now Our Impact in Numbers Transforming Lives Through Education Join thousands who h
--------------------------------------------------

Page 2:
URL: https://banoqabil.pk/news
Text length: 2080
Preview: News & Announcements - Bano Qabil | Bano Qabil Latest Updates News & Announcements Stay Informed. Stay Connected. Stay Inspired. Discover the latest developments, achievements, and milestones from Ban
--------------------------------------------------

Page 3:
URL: https://banoqabil.pk/about
Text length: 4890
Preview: About Us - Bano Qabil | Bano Qabil Alkhidmat Foundation Pakistan Initiative About Bano Qabil Empowering Pakistani Youth Through Education Pakistan's largest free IT training initiative, transforming l
--------------------------------------------------


In [ ]:
extra_urls = [
    "https://banoqabil.pk/courses",
    "https://banoqabil.pk/faq",
    "https://banoqabil.pk/students",
    "https://banoqabil.pk/contact"
]

existing_urls = {page["url"] for page in pages_data}

for url in extra_urls:
    if url not in existing_urls:
        page = scrape_page(url)

        if page:
            print("Collected:", url)

        time.sleep(1)

print("\nTotal pages collected:", len(pages_data))

Collected: https://banoqabil.pk/courses
Collected: https://banoqabil.pk/faq
Collected: https://banoqabil.pk/students
Page skipped: https://banoqabil.pk/contact 404 Client Error: Not Found for url: https://banoqabil.pk/contact

Total pages collected: 6


In [ ]:
for page in pages_data:
    print(page["url"], "—", len(page["text"]), "characters")

https://banoqabil.pk — 5259 characters
https://banoqabil.pk/news — 2080 characters
https://banoqabil.pk/about — 4890 characters
https://banoqabil.pk/courses — 5411 characters
https://banoqabil.pk/faq — 1611 characters
https://banoqabil.pk/students — 1146 characters


In [ ]:
chunks = []

chunk_size = 180
overlap = 30

for page in pages_data:
    words = page["text"].split()

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])

        if len(chunk.strip()) > 50:
            chunks.append({
                "text": chunk,
                "source": page["url"]
            })

print("Total pages:", len(pages_data))
print("Total chunks created:", len(chunks))

Total pages: 6
Total chunks created: 22


In [ ]:
print("First chunk:")
print(chunks[0]["text"][:500])

First chunk:
Bano Qabil - Free IT Training Program Start Your Journey Today Join thousands learning tech skills for free Register Now Our Impact in Numbers Transforming Lives Through Education Join thousands who have already started their journey to success 75,000 + Students Trained 50 + Active Facilities 0 6 Incubation Centres 29 High Tech Courses 100 K+ Registered Tech Employee Our Services What We Offer Comprehensive solutions to transform you from a learner to a professional 01 Advanced IT Training Maste


In [ ]:
from google import genai
from google.genai import types

embedding_model = "gemini-embedding-001"
embedding_dim = 768

def create_embedding(text, task_type="RETRIEVAL_DOCUMENT"):
    result = client.models.embed_content(
        model=embedding_model,
        contents=text,
        config=types.EmbedContentConfig(
            output_dimensionality=embedding_dim,
            task_type=task_type
        )
    )

    return result.embeddings[0].values

test_embedding = create_embedding(chunks[0]["text"])

print("Embedding created successfully!")
print("Embedding dimensions:", len(test_embedding))
print("First 5 values:", test_embedding[:5])

Embedding created successfully!
Embedding dimensions: 768
First 5 values: [-0.006172714, 0.01582993, 0.009908003, -0.03486671, 0.013707084]


In [ ]:
import time

for i, chunk in enumerate(chunks):

    chunk["embedding"] = create_embedding(chunk["text"])

    print(f"Embedding created: {i+1}/{len(chunks)}")

print("\nAll embeddings generated successfully!")

Embedding created: 1/22
Embedding created: 2/22
Embedding created: 3/22
Embedding created: 4/22
Embedding created: 5/22
Embedding created: 6/22
Embedding created: 7/22
Embedding created: 8/22
Embedding created: 9/22
Embedding created: 10/22
Embedding created: 11/22
Embedding created: 12/22
Embedding created: 13/22
Embedding created: 14/22
Embedding created: 15/22
Embedding created: 16/22
Embedding created: 17/22
Embedding created: 18/22
Embedding created: 19/22
Embedding created: 20/22
Embedding created: 21/22
Embedding created: 22/22

All embeddings generated successfully!


In [ ]:
from pinecone import ServerlessSpec

INDEX_NAME = "bano-qabil-rag-02"

existing_indexes = pc.list_indexes().names()

if INDEX_NAME not in existing_indexes:

    pc.create_index(
        name=INDEX_NAME,
        dimension=embedding_dim,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

    print("Pinecone index created!")

else:
    print("Pinecone index already exists!")

index = pc.Index(INDEX_NAME)

Pinecone index already exists!


In [ ]:
import uuid

vectors = []

for i, chunk in enumerate(chunks):

    vectors.append({
        "id": str(uuid.uuid4()),
        "values": chunk["embedding"],
        "metadata": {
            "text": chunk["text"],
            "source": chunk["source"]
        }
    })

for i in range(0, len(vectors), 50):

    index.upsert(
        vectors=vectors[i:i+50]
    )

print("All vectors uploaded successfully!")
print("Total vectors:", len(vectors))

All vectors uploaded successfully!
Total vectors: 22


In [ ]:
import time

time.sleep(3)

stats = index.describe_index_stats()

print("Pinecone Index Statistics:")
print(stats)

Pinecone Index Statistics:
DescribeIndexStatsResponse(dimension=768, total_vector_count=88, metric='cosine', namespaces=1)


In [ ]:
def search_bano_qabil(question, top_k=3):

    question_embedding = create_embedding(
        question,
        task_type="RETRIEVAL_QUERY"
    )

    results = index.query(
        vector=question_embedding,
        top_k=top_k,
        include_metadata=True
    )

    return results.matches


question = "What courses does Bano Qabil offer?"

matches = search_bano_qabil(question)

for match in matches:

    print("Score:", match.score)
    print("Information:", match.metadata["text"][:300])
    print("Source:", match.metadata["source"])
    print("-" * 50)

Score: 0.775776207
Information: Bano Qabil - Free IT Training Program Start Your Journey Today Join thousands learning tech skills for free Register Now Our Impact in Numbers Transforming Lives Through Education Join thousands who have already started their journey to success 75,000 + Students Trained 50 + Active Facilities 0 6 In
Source: https://banoqabil.pk
--------------------------------------------------
Score: 0.775763094
Information: Bano Qabil - Free IT Training Program Start Your Journey Today Join thousands learning tech skills for free Register Now Our Impact in Numbers Transforming Lives Through Education Join thousands who have already started their journey to success 75,000 + Students Trained 50 + Active Facilities 0 6 In
Source: https://banoqabil.pk
--------------------------------------------------
Score: 0.775763094
Information: Bano Qabil - Free IT Training Program Start Your Journey Today Join thousands learning tech skills for free Register Now Our Impact in Numbers

In [ ]:

def ask_bano_qabil(question):

    matches = search_bano_qabil(question, top_k=3)

    if not matches:
        return (
            "I could not find this information on the Bano Qabil website.",
            []
        )

    context = "\n\n".join(
        [match.metadata["text"] for match in matches]
    )

    sources = list(set(
        [match.metadata["source"] for match in matches]
    ))

    prompt = f"""
You are a helpful Bano Qabil AI Assistant.

Answer the user's question using ONLY the provided context.

IMPORTANT RULES:
1. Never invent information.
2. Never provide unsupported claims.
3. If information is missing, clearly say:
   "I could not find this information on the Bano Qabil website."
4. Always provide a complete and meaningful answer.
5. When asked about courses, list the available course names.
6. Use bullet points for lists.
7. Do not return an empty answer.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": "You are a factual Bano Qabil AI assistant. Follow the provided context strictly."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_completion_tokens=1000
    )

    answer = response.choices[0].message.content

    if not answer or not answer.strip():
        answer = (
            "I could not generate an answer. Please try again."
        )

    return answer, sources

In [ ]:
!pip install -q groq

In [ ]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq connected successfully!")

Groq connected successfully!


In [ ]:
answer, sources = ask_bano_qabil(
    "What is Bano Qabil?"
)

print("BANO QABIL AI ASSISTANT")
print(answer)

print("\nSOURCES:")
for source in sources:
    print(source)

BANO QABIL AI ASSISTANT
Bano Qabil is an educational and career‑development platform that has helped thousands of students transform their careers.

SOURCES:
https://banoqabil.pk/courses


In [ ]:
test_questions = [
    "What is Bano Qabil?",
    "Which courses are available?",
    "How can someone register?",
    "Does Bano Qabil guarantee a job to every student?",
    "What is the exact salary after completing a course?",
    "Who is the founder of Bano Qabil?"
]

for question in test_questions:

    print("\nQUESTION:", question)

    answer, sources = ask_bano_qabil(question)

    print("ANSWER:", answer)
    print("SOURCES:", sources)

    print("=" * 60)


QUESTION: What is Bano Qabil?
ANSWER: Bano Qabil is an educational platform that has helped thousands of students transform their careers.
SOURCES: ['https://banoqabil.pk/courses']

QUESTION: Which courses are available?
ANSWER: **Available courses**

- UI/UX Design with Figma: Wireframing, Prototyping, Design Systems & User‑Centered Design — Enhanced with AI Design Tools  
- Summer Camp Web Development with AI: Complete Web Development Course – Learn HTML, CSS & JavaScript from Scratch to Build Responsive Websites  
- E‑Marketing: Digital Content Creation – Automation‑based content business using only a mobile phone, launch and manage monetized social media channels  
- Digital Journalism: Mobile Journalism & Content Creation – Video Production, Editing & Social Media Strategy  
- Digital Marketing: Complete Digital Marketing Course – Social Media, SEO, Google Ads & Freelancing Success  
- Summer Camp Social Media Management: Social Media Manager Course – Facebook, Instagram Marketin

In [ ]:
matches = search_bano_qabil(
    "Which courses are available at Bano Qabil?",
    top_k=3
)

for i, match in enumerate(matches):
    print(f"\n--- MATCH {i+1} ---")
    print("Score:", match.score)
    print("Source:", match.metadata["source"])
    print("Text:", match.metadata["text"])


--- MATCH 1 ---
Score: 0.785671771
Source: https://banoqabil.pk/courses
Text: Course - Facebook, Instagram Marketing, Content Strategy & Analytics Incharge: Haris Alam Duration: 2 months View Details E-Commerce 3 Course s Available E-Commerce Development Mastery Build Online Stores with Shopify, WordPress & WooCommerce - Complete E-Commerce Course Incharge: Ammad Alam Duration: 5 months View Details E-Commerce Marketplace Learn to Sell, Market, and Grow Across Today’s Top E-Commerce Platforms Incharge: Ammad Alam Duration: 5 months View Details Freelancing & Tech Sales Mastery Turn Your Digital Skills into Global Income Incharge: Ammad Alam Duration: 5 months View Details Graphics & Video Animation 2 Course s Available Graphic Designing Learn Graphic Design - Master Adobe Illustrator & Photoshop for Professional Branding Incharge: Haris Alam Duration: 5 months View Details Video Editing & Animations Professional Video Editing Course - Master Adobe Premiere Pro & After Effects for Moti

In [ ]:
answer, sources = ask_bano_qabil(
    "List all available courses at Bano Qabil."
)

print("BANO QABIL AI ASSISTANT")
print(answer)

print("\nSOURCES:")
for source in sources:
    print(source)

BANO QABIL AI ASSISTANT
- **AI for Everyone** – Complete AI Course for Beginners – Understand and Master Generative AI Tools  
- **Backend Development with Node.js** – Node.js, Express.js, MongoDB & REST APIs – Building Production Servers the AI‑Assisted Way  
- **CIT & Programming Foundations** – From Basic Tech Awareness to Programming Mastery – Built for the AI Era  
- **Summer Camp Computer Information and Technology** – Master Essential IT Skills: Google Sheets, Excel, Programming Logic, Prompt Engineering & Git  
- **Cyber Security**  

These are the courses listed in the provided context for Bano Qabil.

SOURCES:
https://banoqabil.pk/courses


In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [ ]:
from google.colab import userdata
from google import genai
from pinecone import Pinecone
from groq import Groq

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

print("Gemini connected!")
print("Pinecone connected!")
print("Groq connected!")

Gemini connected!
Pinecone connected!
Groq connected!
